# Heimdall: which method best spots mule accounts before the bank flags them?

**The question this notebook answers:** *On 1 January 2025, using only the transactions that happened before that date, which accounts will later be flagged as mule accounts?*

We compare four methods on exactly the same data and rules, and report how many mules each one catches **and** how many genuine customers it wrongly flags.

- **Data:** RBIH × IIT Delhi National Fraud Prevention Challenge (NFPC) Phase-2 dataset, Kaggle `abhyudayrbih/rbih-nfpc-phase-2`, licence CC BY-NC-SA 4.0 (non-commercial). The data is **synthetic**: results describe this dataset, not a real bank.
- **How to run:** attach the dataset, set *Accelerator* to *None*, then **Run All**. The full run takes roughly 20–30 minutes. For a quick trial, set `N_FILES = 20` in the next cell first.
- The steps follow Figure 2 in the project README.

## Settings
`CUTOFF` is the "today" of our test: the methods may only see transactions **before** it. `RECENT_DAYS` is the window used for the "last 30 days" numbers.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd

CUTOFF = pd.Timestamp('2025-01-01')
RECENT_DAYS = 30
N_FILES = None                 # None = all 396 files; e.g. 20 for a quick trial run
INPUT_ROOT = '/kaggle/input'
OUTPUT_DIR = '/kaggle/working'
RECENT = CUTOFF - pd.Timedelta(days=RECENT_DAYS)

## Step 0. Find the data
Kaggle puts attached datasets under `/kaggle/input`. We search for the labels file so the notebook works whatever the folder is called.

In [ ]:
DATA = os.path.dirname(glob.glob(INPUT_ROOT + '/**/train_labels.parquet', recursive=True)[0])
parts = sorted(glob.glob(DATA + '/transactions/batch-*/part_*.parquet'))
if N_FILES: parts = parts[:N_FILES]
print('data folder:', DATA)
print(len(parts), 'transaction files')          # 396 for the full dataset

## Look at one file before doing anything
Always look at the raw data first. `txn_type`: **C** = money in (credit), **D** = money out (debit). `channel`: **UPC/UPD** = UPI credit/debit.

In [ ]:
one = pd.read_parquet(parts[0])
print(one.shape)
print(one['txn_type'].value_counts())
print(one['channel'].value_counts().head(5))
one.head()

## Step 2. Summarise one file, using only transactions before the cutoff
For each file we build two small tables:

- **`acc`**, one row per account: number of transactions, money in, money out, money in/out in the last 30 days, number of UPI transactions, first and last transaction time.
- **`con`**, one row per *(account, contact)*: did money ever come **in** from this contact, did money ever go **out** to them, and when we first saw them.

Two rules: we drop everything on or after the cutoff (no peeking at the future), and we drop `BANK_INTEREST` rows (the bank's own monthly interest postings, not customer behaviour).

In [ ]:
COLS = ['account_id', 'transaction_timestamp', 'channel', 'amount', 'txn_type', 'counterparty_id']

def summarise_file(path):
    t = pd.read_parquet(path, columns=COLS)
    t['ts'] = pd.to_datetime(t['transaction_timestamp'], format='ISO8601')        # some files use a space instead of 'T'
    t = t[(t['ts'] < CUTOFF) & (t['counterparty_id'] != 'BANK_INTEREST')].copy()
    t['amt'] = t['amount'].abs()
    t['is_in'] = t['txn_type'] == 'C'
    t['is_out'] = ~t['is_in']
    t['money_in'] = t['amt'].where(t['is_in'], 0.0)
    t['money_out'] = t['amt'].where(t['is_out'], 0.0)
    recent = t['ts'] >= RECENT
    t['in_30d'] = t['money_in'].where(recent, 0.0)
    t['out_30d'] = t['money_out'].where(recent, 0.0)
    t['upi'] = t['channel'].isin(['UPC', 'UPD'])
    acc = t.groupby('account_id').agg(n_txn=('amt', 'size'), money_in=('money_in', 'sum'), money_out=('money_out', 'sum'),
                                      in_30d=('in_30d', 'sum'), out_30d=('out_30d', 'sum'), n_upi=('upi', 'sum'),
                                      first_ts=('ts', 'min'), last_ts=('ts', 'max'))
    con = t.groupby(['account_id', 'counterparty_id']).agg(has_in=('is_in', 'max'), has_out=('is_out', 'max'),
                                                          first_seen=('ts', 'min'))
    return acc, con

a, c = summarise_file(parts[0])
print(a.shape, c.shape)
a.head()

## Step 1. Join each account's transactions from ALL files
Some accounts have their transactions spread over two or three files (in the full dataset: about 7% of accounts, and about half of the mules). If we summarised each file separately, those accounts would look smaller than they really are. So we summarise every file, then **add the pieces together per account**.

In [ ]:
accs, cons = [], []
start = time.time()
for i, p in enumerate(parts):
    a, c = summarise_file(p)
    accs.append(a); cons.append(c)
    if i % 25 == 0:
        print(i, 'files done,', round(time.time() - start), 'seconds')

all_accs = pd.concat(accs)
print('accounts that appear in more than one file:', all_accs.index.duplicated().sum())

A = all_accs.groupby(level=0).agg({'n_txn': 'sum', 'money_in': 'sum', 'money_out': 'sum', 'in_30d': 'sum',
                                   'out_30d': 'sum', 'n_upi': 'sum', 'first_ts': 'min', 'last_ts': 'max'})
C = pd.concat(cons).groupby(level=[0, 1]).agg({'has_in': 'max', 'has_out': 'max', 'first_seen': 'min'})
print(len(A), 'accounts;', len(C), 'account-contact pairs')

## Step 3. Turn each account into numbers (features)
| Feature | Plain meaning |
|---|---|
| `n_txn` | number of transactions |
| `n_contacts` | number of different people/accounts it dealt with |
| `two_way_share` | share of contacts where money went **both** ways (genuine relationships tend to be two-way) |
| `contacts_in`, `contacts_out` | how many contacts sent money in / received money out |
| `money_in`, `money_out` | total money received / sent |
| `out_in_ratio` | money sent ÷ money received (a pass-through account sends on most of what it gets) |
| `upi_share` | share of transactions made by UPI |
| `account_age_days` | days since the account was opened |
| `active_days` | days since its first transaction |
| `days_since_last_txn` | days since its latest transaction |
| `txn_per_day` | transactions per active day |
| `in_30d`, `out_30d`, `out_in_ratio_30d` | the same money numbers for the last 30 days only |
| `new_contacts_30d` | contacts first seen in the last 30 days |

`rural_branch` is kept **only** for the fairness check at the end. It is **not** given to the models.

In [ ]:
C['two_way'] = C['has_in'] & C['has_out']
C['new_30d'] = C['first_seen'] >= RECENT
G = C.groupby(level=0).agg(n_contacts=('two_way', 'size'), two_way_contacts=('two_way', 'sum'),
                           contacts_in=('has_in', 'sum'), contacts_out=('has_out', 'sum'),
                           new_contacts_30d=('new_30d', 'sum'))
info = pd.read_parquet(DATA + '/accounts.parquet', columns=['account_id', 'account_opening_date', 'rural_branch']).set_index('account_id')
A = A.join(G).join(info)

F = pd.DataFrame(index=A.index)
F['n_txn'] = A['n_txn']
F['n_contacts'] = A['n_contacts']
F['two_way_share'] = A['two_way_contacts'] / A['n_contacts']
F['contacts_in'] = A['contacts_in']
F['contacts_out'] = A['contacts_out']
F['money_in'] = A['money_in']
F['money_out'] = A['money_out']
F['out_in_ratio'] = A['money_out'] / (A['money_in'] + 1)
F['upi_share'] = A['n_upi'] / A['n_txn']
F['account_age_days'] = (CUTOFF - pd.to_datetime(A['account_opening_date'], errors='coerce')).dt.days
F['active_days'] = (CUTOFF - A['first_ts']).dt.days
F['days_since_last_txn'] = (CUTOFF - A['last_ts']).dt.days
F['txn_per_day'] = A['n_txn'] / F['active_days'].clip(lower=1)
F['in_30d'] = A['in_30d']
F['out_30d'] = A['out_30d']
F['out_in_ratio_30d'] = A['out_30d'] / (A['in_30d'] + 1)
F['new_contacts_30d'] = A['new_contacts_30d']
F['rural_branch'] = A['rural_branch'] == 'Y'
F.describe().T

## Step 4. Label each account
- **1** = a mule that the bank flagged **on or after** the cutoff (not yet caught on 1 Jan 2025: these are the ones we want to find early)
- **0** = a genuine account
- Mules flagged **before** the cutoff are dropped: they were already caught, so finding them proves nothing.

Only accounts from `train_labels` have answers, so only they are used.

In [ ]:
lab = pd.read_parquet(DATA + '/train_labels.parquet', columns=['account_id', 'is_mule', 'mule_flag_date']).set_index('account_id')
lab['flag'] = pd.to_datetime(lab['mule_flag_date'], errors='coerce')
D = F.join(lab, how='inner')
already_caught = (D['is_mule'] == 1) & (D['flag'] < CUTOFF)
print('mules flagged before the cutoff (dropped):', already_caught.sum())
D = D[~already_caught].copy()
D['y'] = D['is_mule'].astype(int)
print(len(D), 'accounts,', D['y'].sum(), 'mules =', round(D['y'].mean() * 100, 2), '%')
D.to_parquet(OUTPUT_DIR + '/benchmark_table.parquet')

## Steps 5–8. The benchmark rules
- **Same features** for every method.
- **Default settings** for every method (no extra tuning for anyone, so no method gets an unfair advantage).
- **5 different random splits**: 70% of accounts to learn from, 30% to test on. Every split keeps the same share of mules, and test accounts are never seen during learning.
- We report the **average and the spread** over the 5 splits. If two methods differ by less than the spread, treat them as a tie.

**The four methods**
1. **Simple rule:** score = (1 − two-way share) × log(number of contacts). No learning; this is the kind of rule banks already use.
2. **Logistic regression:** gives each feature a weight and adds them up.
3. **Random forest:** hundreds of yes/no decision trees that vote.
4. **Gradient boosting (LightGBM):** trees built one after another, each fixing the previous ones' mistakes.

**The scores**
- **ROC-AUC:** pick one mule and one genuine account at random; how often does the mule get the higher score? 0.5 = coin toss, 1 = perfect.
- **PR-AUC:** when the method raises alerts, how many are real mules? A coin toss scores about 0.02 here (the mule rate).
- **Mules caught at 1 false alarm per 100 (or per 1,000) genuine accounts:** the number a bank manager cares about.
- **Time per prediction:** time to score one account, which is what the real-time check needs.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import lightgbm as lgb

FEATS = ['n_txn', 'n_contacts', 'two_way_share', 'contacts_in', 'contacts_out', 'money_in', 'money_out', 'out_in_ratio', 'upi_share',
         'account_age_days', 'active_days', 'days_since_last_txn', 'txn_per_day', 'in_30d', 'out_30d', 'out_in_ratio_30d', 'new_contacts_30d']
X = D[FEATS].astype(float)
y = D['y'].to_numpy()
Xlog = np.sign(X) * np.log1p(np.abs(X))          # log scale helps the linear model with large money amounts

def recall_at_false_alarm(y_true, score, rate):
    fpr, tpr, _ = roc_curve(y_true, score)
    return float(np.interp(rate, fpr, tpr))

METHODS = {
    'Simple rule': None,
    'Logistic regression': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight='balanced')),
    'Random forest': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, class_weight='balanced_subsample',
                                                    n_jobs=2, random_state=0),
    'LightGBM': lambda: lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, n_jobs=2, verbose=-1, random_state=0),
}

### Run the benchmark (the longest cell, about 5 minutes)
It also runs two checks:
- **LightGBM without `two_way_share`:** in this synthetic data almost every genuine account is two-way, which may not be true in a real bank. If the result barely drops without it, the model does not depend on that one pattern.
- **Fairness:** among genuine accounts, how often are rural and urban customers wrongly flagged at the same setting?

In [ ]:
rows, fair, people = [], [], []
splits = StratifiedShuffleSplit(n_splits=5, test_size=0.3, random_state=42)
for k, (tr, te) in enumerate(splits.split(X, y)):
    med = X.iloc[tr].median()                                  # fill any gaps with the TRAINING median only
    Xk, Xlk = X.fillna(med), Xlog.fillna(np.sign(med) * np.log1p(np.abs(med)))
    for name, make in METHODS.items():
        Z = Xlk if name == 'Logistic regression' else Xk
        if make is None:
            score = ((1 - Xk['two_way_share']) * np.log1p(Xk['n_contacts'])).to_numpy()[te]; model = None
        else:
            model = make().fit(Z.iloc[tr], y[tr]); score = model.predict_proba(Z.iloc[te])[:, 1]
        ms = np.nan
        if model is not None:                                  # time to score ONE account, median of 300 tries
            times = []
            for i in te[:300]:
                t1 = time.perf_counter(); model.predict_proba(Z.iloc[[i]]); times.append(time.perf_counter() - t1)
            ms = np.median(times) * 1000
        yt = y[te]
        rows.append({'method': name, 'split': k, 'ROC-AUC': roc_auc_score(yt, score), 'PR-AUC': average_precision_score(yt, score),
                     'caught @1/1000 false alarms': recall_at_false_alarm(yt, score, 0.001),
                     'caught @1/100 false alarms': recall_at_false_alarm(yt, score, 0.01), 'ms per prediction': ms})
        if name == 'LightGBM':
            th = np.quantile(score[yt == 0], 0.99)             # setting that wrongly flags 1 in 100 genuine accounts
            genuine = D.iloc[te][yt == 0]; flagged = score[yt == 0] >= th
            fair.append({'rural': flagged[genuine['rural_branch'].to_numpy()].mean(), 'urban': flagged[~genuine['rural_branch'].to_numpy()].mean()})
            people.append({'test accounts': len(te), 'mules': int(yt.sum()), 'genuine wrongly flagged': int(flagged.sum()),
                           'mules caught': int((score[yt == 1] >= th).sum())})
    Xa = Xk.drop(columns=['two_way_share'])
    m = METHODS['LightGBM']().fit(Xa.iloc[tr], y[tr]); score = m.predict_proba(Xa.iloc[te])[:, 1]; yt = y[te]
    rows.append({'method': 'LightGBM without two_way_share (check)', 'split': k, 'ROC-AUC': roc_auc_score(yt, score),
                 'PR-AUC': average_precision_score(yt, score), 'caught @1/1000 false alarms': recall_at_false_alarm(yt, score, 0.001),
                 'caught @1/100 false alarms': recall_at_false_alarm(yt, score, 0.01), 'ms per prediction': np.nan})
    print('split', k, 'done')
R = pd.DataFrame(rows)

## Results: average ± spread over the 5 splits

In [ ]:
mean = R.groupby('method', sort=False).mean(numeric_only=True).drop(columns='split')
std = R.groupby('method', sort=False).std(numeric_only=True).drop(columns='split')
table = mean.round(3).astype(str) + ' ± ' + std.round(3).astype(str)
R.to_csv(OUTPUT_DIR + '/benchmark_results_by_split.csv', index=False)
table.to_csv(OUTPUT_DIR + '/benchmark_summary.csv')
table

## The same result, in people
For one test split, at the setting where LightGBM wrongly flags 1 in 100 genuine accounts:

In [ ]:
p = people[0]
print(f"Test accounts: {p['test accounts']:,}, of which {p['mules']} are mules")
print(f"Genuine customers wrongly flagged: {p['genuine wrongly flagged']}")
print(f"Mules caught: {p['mules caught']} of {p['mules']} ({p['mules caught'] / p['mules']:.0%})")
print(f"Share of alerts that are real mules: {p['mules caught'] / (p['mules caught'] + p['genuine wrongly flagged']):.0%}")

## Fairness check
Among genuine customers, how often is each group wrongly flagged at the same setting? A large gap means one group would face more wrongful holds, so those flags should go to a person before any hold is placed.

In [ ]:
f = pd.DataFrame(fair).mean()
print(f"Genuine customers wrongly flagged: rural {f['rural']:.2%} vs urban {f['urban']:.2%} (ratio {f['rural'] / f['urban']:.1f}x)")

## What this notebook does NOT show
- **Real banks:** the data is synthetic. Some patterns (for example almost every genuine account being two-way) may not exist in real banks.
- **Labels are the bank's own flags,** and most mules in this dataset were flagged in one sweep (April–June 2025). Undetected mules count as "genuine", so all scores here are approximate.
- **One cutoff date.** A different date could give somewhat different numbers.
- **No tuning,** by design, so all methods get equal effort. Tuned methods may score higher.
- **Speed** here is the time to score one account. The full real-time check (updating the account's numbers on every transaction, then scoring) is the next stage of the project.